In [3]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

# ==========================================
# 1. ACQUISITION DES DONNÉES
# ==========================================
file_path = 'data/rawdata/finaldata.csv'
df = pd.read_csv(file_path)

print(f"Nombre total d'échantillons originaux : {len(df)}") 

# ==========================================
# 2. NETTOYAGE DES DONNÉES (CLEANING)
# ==========================================
def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Suppression des balises HTML
    text = re.sub(r'<.*?>', ' ', text)

    # Suppression des retours à la ligne
    text = text.replace('\n', ' ').replace('\r', ' ')

    # Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['body_clean'] = df['body'].apply(clean_text)

# ==========================================
# 3. CONSERVATION DES CATÉGORIES ORIGINALES
# ==========================================
df['label'] = df['queue']

print("\n--- Distribution des classes ---")
print(df['label'].value_counts())

print("\n--- Pourcentage par classe ---")
print((df['label'].value_counts(normalize=True) * 100).round(2))

# ==========================================
# 4. SUPPRESSION DES LIGNES VIDES
# ==========================================
df = df[df['body_clean'].str.len() > 0]


# ==========================================
# 4. SÉPARATION STRATIFIÉE 80/10/10
# ==========================================
# Étape 1 : Séparer en Entraînement (80%) et Reste (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['body_clean'], 
    df['label'], 
    test_size=0.20, 
    stratify=df['label'], # Stratification cruciale ici
    random_state=42
)

# Étape 2 : Séparer le Reste (20%) en Validation (10%) et Test (10%)
# Puisque test_size=0.5 sur les 20% restants, cela donne 10% du dataset total chacun.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, 
    y_temp, 
    test_size=0.50, 
    stratify=y_temp, 
    random_state=42
)

print("\n--- Tailles des sous-ensembles ---")
print(f"Train: {len(X_train)} échantillons ({len(X_train)/len(df)*100:.0f}%)")
print(f"Validation: {len(X_val)} échantillons ({len(X_val)/len(df)*100:.0f}%)")
print(f"Test: {len(X_test)} échantillons ({len(X_test)/len(df)*100:.0f}%)")

# Sauvegarde des datasets propres
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
val_df = pd.DataFrame({'text': X_val, 'label': y_val})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

train_df.to_csv('data/train.csv', index=False)
val_df.to_csv('data/val.csv', index=False)
test_df.to_csv('data/test.csv', index=False)


Nombre total d'échantillons originaux : 4416

--- Distribution des classes ---
label
Support                 2094
Billing and Payments    1302
Feature Request         1020
Name: count, dtype: int64

--- Pourcentage par classe ---
label
Support                 47.42
Billing and Payments    29.48
Feature Request         23.10
Name: proportion, dtype: float64

--- Tailles des sous-ensembles ---
Train: 3532 échantillons (80%)
Validation: 442 échantillons (10%)
Test: 442 échantillons (10%)
